# LangChain + OpenAI Text Summarization from CSV

This notebook demonstrates an end-to-end mini project for:

- reading text data from a CSV
- masking sensitive data (PII)
- cleaning the text
- generating summaries using **LangChain + OpenAI**
- checking summary quality with simple evaluation metrics
- preparing the same logic for a **Streamlit app**

## Dataset
The included file is `ecommerce_text_data.csv`.

## Environment variables
Create a `.env` file with:

```bash
OPENAI_API_KEY=your_api_key_here
```


In [ ]:
# # Install once if needed
# !pip install -U pandas python-dotenv langchain-openai openai streamlit

In [1]:
import os
import re
import json
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in environment or .env file")

c:\python_vs\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
df = pd.read_csv("ecommerce_text_data.csv")
df.head()

,order_id,customer_name,email,phone,city,category,product,order_value,status,sentiment,text,reference_summary
0,ORD10001,Anil Joshi,anil.joshi87@example.com,+91-99234-81482,Delhi,fashion,running shoes,6200.70,refunded,neutral,Customer Anil Joshi from Delhi placed order OR...,Anil ordered running shoes. Order status is re...
1,ORD10002,Priya Reddy,priya.reddy72@gmail.com,+91-93462-95181,Hyderabad,accessories,laptop backpack,6429.39,refunded,positive,Customer Priya Reddy from Hyderabad placed ord...,Priya ordered laptop backpack. Order status is...
2,ORD10003,Divya Menon,divya.menon98@gmail.com,+91-92876-65392,Chennai,accessories,laptop backpack,20387.22,in_transit,negative,Customer Divya Menon from Chennai placed order...,Divya ordered laptop backpack. Order status is...
3,ORD10004,Neha Singh,neha.singh46@outlook.com,+91-89782-44671,Bengaluru,furniture,office chair,3043.45,delivered,neutral,Customer Neha Singh from Bengaluru placed orde...,Neha ordered office chair. Order status is del...
4,ORD10005,Aarav Sharma,aarav.sharma74@gmail.com,+91-93087-19116,Hyderabad,fitness,water bottle,14158.38,delivered,neutral,Customer Aarav Sharma from Hyderabad placed or...,Aarav ordered water bottle. Order status is de...


## 1. Data cleaning and PII masking

We will:
- remove extra spaces
- normalize text
- mask emails
- mask phone numbers
- mask order IDs
- remove customer names from the text using the CSV column


In [3]:
def mask_pii(text, customer_name):
    text = str(text).lower().strip()
    text = re.sub(r"\S+@\S+", "[EMAIL]", text)
    text = re.sub(r"\+?\d[\d\s\-]{8,}\d", "[PHONE]", text)
    text = re.sub(r"ord\d+", "[ORDER_ID]", text, flags=re.IGNORECASE)
    if pd.notna(customer_name):
        text = re.sub(customer_name.lower(), "[CUSTOMER_NAME]", text)
    text = re.sub(r"\s+", " ", text)
    return text


df["masked_text"] = df.apply(
    lambda row: mask_pii(row["text"], row["customer_name"]),
    axis=1
)

df[["text", "masked_text"]].head()

,text,masked_text
0,Customer Anil Joshi from Delhi placed order OR...,customer [CUSTOMER_NAME] from delhi placed ord...
1,Customer Priya Reddy from Hyderabad placed ord...,customer [CUSTOMER_NAME] from hyderabad placed...
2,Customer Divya Menon from Chennai placed order...,customer [CUSTOMER_NAME] from chennai placed o...
3,Customer Neha Singh from Bengaluru placed orde...,customer [CUSTOMER_NAME] from bengaluru placed...
4,Customer Aarav Sharma from Hyderabad placed or...,customer [CUSTOMER_NAME] from hyderabad placed...


## 2. Build the LangChain summarization pipeline

We use `ChatOpenAI` from the `langchain-openai` package, which is the current integration package for OpenAI models in LangChain. 


In [4]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.2
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert ecommerce analyst who summarizes customer orders."),
    ("human", "Summarize the following order clearly and concisely:\n\n{text}")
])

In [5]:
def summarize_text(text):
    chain = prompt | llm
    response = chain.invoke({"text": text})
    return response.content

## 3. Summarize a subset or all rows

To control token usage, start with a subset and then scale.


In [6]:
sample_df = df.sample(10, random_state=42).copy()

sample_df["generated_summary"] = sample_df["masked_text"].apply(summarize_text)

sample_df[["masked_text", "generated_summary"]].head()

,masked_text,generated_summary
83,customer [CUSTOMER_NAME] from delhi placed ord...,Customer [CUSTOMER_NAME] from Delhi placed ord...
53,customer [CUSTOMER_NAME] from pune placed orde...,Customer [CUSTOMER_NAME] from Pune ordered 3 u...
70,customer [CUSTOMER_NAME] from hyderabad placed...,Customer [CUSTOMER_NAME] from Hyderabad ordere...
45,customer [CUSTOMER_NAME] from chennai placed o...,Customer [CUSTOMER_NAME] from Chennai ordered ...
44,customer [CUSTOMER_NAME] from chennai placed o...,Customer [CUSTOMER_NAME] from Chennai ordered ...


## 4. Accuracy / quality check

Text summarization does not have a single perfect accuracy metric.  
Here we use two simple proxy checks:

1. **Token Overlap F1** between generated summary and reference summary  
2. **Critical Keyword Coverage** for important fields like product, status, and sentiment

This is not a perfect evaluation, but it gives a practical quality estimate.


In [7]:
def token_overlap_f1(pred, ref):
    pred_tokens = set(str(pred).lower().split())
    ref_tokens = set(str(ref).lower().split())
    if not pred_tokens or not ref_tokens:
        return 0.0
    overlap = pred_tokens & ref_tokens
    precision = len(overlap) / len(pred_tokens)
    recall = len(overlap) / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


def keyword_coverage(pred, row):
    keywords = [
        row["product"],
        row["status"],
        row["sentiment"]
    ]
    pred_text = str(pred).lower()
    hits = sum(1 for k in keywords if str(k).lower() in pred_text)
    return hits / len(keywords)


sample_df["token_overlap_f1"] = sample_df.apply(
    lambda r: token_overlap_f1(r["generated_summary"], r["reference_summary"]), axis=1
)

sample_df["keyword_coverage"] = sample_df.apply(
    lambda r: keyword_coverage(r["generated_summary"], r), axis=1
)

sample_df["quality_score_percent"] = (
    sample_df["token_overlap_f1"] * 0.7 +
    sample_df["keyword_coverage"] * 0.3
) * 100

In [8]:
print("Average token overlap F1:", round(sample_df["token_overlap_f1"].mean(), 3))
print("Average keyword coverage:", round(sample_df["keyword_coverage"].mean(), 3))
print("Average quality score (%):", round(sample_df["quality_score_percent"].mean(), 2))

Average token overlap F1: 0.366
Average keyword coverage: 0.9
Average quality score (%): 52.61


## 5. Run on the full dataset

Uncomment the next cell when ready.


In [9]:
full_df = df.copy()
full_df["generated_summary"] = full_df["masked_text"].apply(summarize_text)
full_df["token_overlap_f1"] = full_df.apply(
    lambda row: token_overlap_f1(row["generated_summary"], row["reference_summary"]), axis=1
)
full_df["keyword_coverage"] = full_df.apply(
    lambda row: keyword_coverage(row["generated_summary"], row), axis=1
)
full_df["quality_score_percent"] = ((full_df["token_overlap_f1"] * 0.7) + (full_df["keyword_coverage"] * 0.3)) * 100
full_df.to_csv("summarized_output.csv", index=False)
full_df.head()

,order_id,customer_name,email,phone,city,category,product,order_value,status,sentiment,text,reference_summary,masked_text,generated_summary,token_overlap_f1,keyword_coverage,quality_score_percent
0,ORD10001,Anil Joshi,anil.joshi87@example.com,+91-99234-81482,Delhi,fashion,running shoes,6200.70,refunded,neutral,Customer Anil Joshi from Delhi placed order OR...,Anil ordered running shoes. Order status is re...,customer [CUSTOMER_NAME] from delhi placed ord...,Customer [CUSTOMER_NAME] from Delhi ordered 1 ...,0.392157,1.000000,57.450980
1,ORD10002,Priya Reddy,priya.reddy72@gmail.com,+91-93462-95181,Hyderabad,accessories,laptop backpack,6429.39,refunded,positive,Customer Priya Reddy from Hyderabad placed ord...,Priya ordered laptop backpack. Order status is...,customer [CUSTOMER_NAME] from hyderabad placed...,Customer [CUSTOMER_NAME] from Hyderabad ordere...,0.415094,1.000000,59.056604
2,ORD10003,Divya Menon,divya.menon98@gmail.com,+91-92876-65392,Chennai,accessories,laptop backpack,20387.22,in_transit,negative,Customer Divya Menon from Chennai placed order...,Divya ordered laptop backpack. Order status is...,customer [CUSTOMER_NAME] from chennai placed o...,Customer [CUSTOMER_NAME] from Chennai ordered ...,0.400000,0.666667,48.000000
3,ORD10004,Neha Singh,neha.singh46@outlook.com,+91-89782-44671,Bengaluru,furniture,office chair,3043.45,delivered,neutral,Customer Neha Singh from Bengaluru placed orde...,Neha ordered office chair. Order status is del...,customer [CUSTOMER_NAME] from bengaluru placed...,Customer [CUSTOMER_NAME] from Bengaluru ordere...,0.367347,1.000000,55.714286
4,ORD10005,Aarav Sharma,aarav.sharma74@gmail.com,+91-93087-19116,Hyderabad,fitness,water bottle,14158.38,delivered,neutral,Customer Aarav Sharma from Hyderabad placed or...,Aarav ordered water bottle. Order status is de...,customer [CUSTOMER_NAME] from hyderabad placed...,Customer [CUSTOMER_NAME] ordered 3 water bottl...,0.384615,1.000000,56.923077


## 6. What the Streamlit app will do

The Streamlit app will:

- upload a CSV
- let the user pick the text column
- clean and mask the data
- call OpenAI through LangChain
- show summaries and evaluation scores
- allow CSV download
